# Kenya Gazette Parser — Quickstart

This notebook shows how to extract structured data from Kenya Gazette PDFs.

In [ ]:
## 1. Installation

```bash
pip install git+https://github.com/RonKG/docling_spatial_pdfs.git
```

## 2. Parse a PDF

In [ ]:
import os
from pathlib import Path

# Optional: set this explicitly if auto-discovery does not find your PDF.
# PDF_PATH = r"C:\path\to\Kenya Gazette.pdf"
try:
    PDF_PATH
except NameError:
    PDF_PATH = None


def find_gazette_pdf() -> Path:
    explicit_path = PDF_PATH or os.getenv("KENYA_GAZETTE_PDF")
    if explicit_path:
        path = Path(explicit_path).expanduser()
        if path.is_file():
            return path
        raise FileNotFoundError(f"PDF path does not point to a file: {path}")

    search_dirs = [
        Path.cwd(),
        Path.cwd() / "pdfs",
        Path.cwd().parent / "pdfs",
        Path.cwd().parent,
    ]
    for directory in search_dirs:
        if directory.is_dir():
            matches = sorted(directory.glob("*.pdf"))
            if matches:
                return matches[0]

    raise FileNotFoundError(
        "No PDF found. Set PDF_PATH to a local Kenya Gazette PDF path, "
        "or set the KENYA_GAZETTE_PDF environment variable."
    )


pdf_path = find_gazette_pdf()
print(f"PDF: {pdf_path}")

### Optional LLM Policy

`GazetteConfig.llm` is the public configuration surface for LLM-assisted stages. In the current 1.0 package these options are accepted and carried through the API, but the actual LLM validation stages are not invoked yet. Keep `mode="disabled"` for fully offline runs, or use `mode="optional"` to make the notebook ready for LLM-backed validation once those stages are wired.

In [ ]:
from kenya_gazette_parser import GazetteConfig, LLMPolicy

config = GazetteConfig(
    llm=LLMPolicy(
        mode="optional",
        model="gpt-4o-mini",
        api_key_env="OPENAI_API_KEY",
        stages={"validate_notices": True},
        cache_dir=Path(".llm_cache"),
    )
)

print(f"LLM mode: {config.llm.mode}")
print(f"LLM model: {config.llm.model}")
print(f"API key env var: {config.llm.api_key_env}")
print(f"Enabled LLM stages: {config.llm.stages}")

if not os.getenv(config.llm.api_key_env):
    print(
        f"Note: {config.llm.api_key_env} is not set. "
        "That is fine for today's parser because LLM stages are not invoked yet."
    )

In [ ]:
from kenya_gazette_parser import parse_file

envelope = parse_file(pdf_path, config=config)

print(f"Library version: {envelope.library_version}")
print(f"Notices found: {len(envelope.notices)}")
print(f"Corrigenda found: {len(envelope.corrigenda)}")

## 3. Explore the Envelope

In [ ]:
# Issue metadata
issue = envelope.issue
print(f"Volume: {issue.volume}")
print(f"Issue: {issue.issue_no}")
print(f"Date: {issue.publication_date}")
print(f"Issue ID: {issue.gazette_issue_id}")

In [ ]:
# Look at the first notice
notice = envelope.notices[0]
title = " ".join(notice.title_lines) or notice.gazette_notice_header or "(untitled)"
print(f"Notice ID: {notice.notice_id}")
print(f"Title: {title}")
print(f"Confidence: {notice.confidence_scores.composite:.2%}")
print(f"\nFirst 200 chars of text:")
print(notice.gazette_notice_full_text[:200])

In [ ]:
# Document-level confidence
conf = envelope.document_confidence
print(f"Mean composite score: {conf.mean_composite:.2%}")
print(f"OCR quality: {conf.ocr_quality:.2%}")

## 4. Write Output Files

In [ ]:
from kenya_gazette_parser import write_envelope

# Write all output files to a directory. pdf_path is reused from the parse cell
# because some raw-text bundles need access to the original PDF.
written = write_envelope(
    envelope,
    out_dir="./output_demo",
    pdf_path=pdf_path,
)

print("Files written:")
for name, path in written.items():
    print(f"  {name}: {path}")

## 5. Validate Against JSON Schema

In [ ]:
from kenya_gazette_parser import validate_envelope_json

# Convert envelope to dict and validate
data = envelope.model_dump(mode="json")
is_valid = validate_envelope_json(data)
print(f"Envelope is valid: {is_valid}")

## 6. Export to JSON

In [ ]:
import json

# Save as JSON file
with open("./output_demo/envelope.json", "w") as f:
    json.dump(envelope.model_dump(mode="json"), f, indent=2)

print("Saved to ./output_demo/envelope.json")

---

**That's it!** For more details, see:
- [README.md](../README.md) — quickstart guide
- [docs/library-contract-v1.md](../docs/library-contract-v1.md) — full API reference